This project implements an asynchronous, multi-agent research workflow that automatically plans web searches, retrieves information, and compiles a three-paragraph Markdown report based on a user's query. Additionally, it uses specialized AI agents as input and output guardrails to ensure that both the user's prompt and the final generated response remain strictly focused on technology, software, or business topics.

In [ ]:
import asyncio
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from agents import Agent, Runner, trace, WebSearchTool, input_guardrail, output_guardrail, GuardrailFunctionOutput
from agents.model_settings import ModelSettings
from IPython.display import display, Markdown

In [ ]:
load_dotenv(override=True)

Define the Guardrail

In [ ]:
class TopicCheckOutput(BaseModel):
    is_tech_or_business: bool
    topic: str

In [ ]:
guardrail_agent = Agent( 
    name="Topic Checker",
    instructions="Check if the user's query is related to technology, software, or business. Return true if it is, false otherwise.",
    output_type=TopicCheckOutput,
    model="gpt-4o-mini"
)

In [ ]:
@input_guardrail
async def guardrail_against_off_topic(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_valid_topic = result.final_output.is_tech_or_business
    return GuardrailFunctionOutput(
        output_info={"valid_topic": is_valid_topic}, 
        tripwire_triggered=not is_valid_topic
    )

CONVERSATIONAL OUTPUT GUARDRAIL

In [ ]:
class PartnerBehaviorCheck(BaseModel):
    is_strictly_tech_software_business_related: bool = Field(
        description="True if the response strictly discusses technology, software, business, or politely apologizes and redirects non-coding topics. False if it engages in off-topic conversations."
    )

In [ ]:
behavior_checker_agent = Agent(
    name="Behavior Quality Checker",
    instructions="Review the proposed response from the writer agent. Check if the writer's agent response is related to technology, software, or business. Return true if it is, false otherwise.",
    output_type=PartnerBehaviorCheck,
    model="gpt-4o-mini"
)

In [ ]:
@output_guardrail
async def guardrail_partner_behavior(ctx, agent, message):
    result = await Runner.run(behavior_checker_agent, message, context=ctx.context)
    checks = result.final_output
    is_valid = (
        checks.is_strictly_tech_software_business_related
    )
    return GuardrailFunctionOutput(
        output_info={"checks": checks.model_dump(), "passed_all": is_valid},
        tripwire_triggered=not is_valid 
    )

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")

In [ ]:
class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of exactly 2 web searches to perform to best answer the query.")

In [ ]:
planner_agent = Agent(
    name="Planner Agent",
    instructions="You are a helpful research assistant. Given a broad query, come up with exactly 2 specific web searches to perform.",
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
    input_guardrails=[guardrail_against_off_topic] # Attach guardrail here
)

Define the Search Agent

In [ ]:
search_agent = Agent(
    name="Search Agent",
    instructions="You are a research assistant. Search the web for the given term and produce a concise 1-paragraph summary of the main points. Ignore fluff.",
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

Define the Writer Agent

In [ ]:
writer_agent = Agent(
    name="Writer Agent",
    instructions="You are a senior researcher. Combine the original query and the summarized search results into a clean, 3-paragraph Markdown report.",
    model="gpt-4o-mini",
    output_guardrails=[guardrail_partner_behavior]
)

Orchestrate the Workflow

In [ ]:
async def run_research_workflow(user_query: str):
    print(f"Starting research on: {user_query}")
    
    with trace("Full Research Workflow"):
        # 1. Plan the searches (Guardrail runs automatically here)
        try:
            plan_result = await Runner.run(planner_agent, user_query)
            search_plan = plan_result.final_output
        except Exception as e:
            print(f"Workflow stopped: {e}")
            return
            
        print(f"Generated {len(search_plan.searches)} search queries.")
        
        # 2. Execute searches concurrently
        tasks = []
        for item in search_plan.searches:
            input_text = f"Search term: {item.query}\nReason: {item.reason}"
            tasks.append(Runner.run(search_agent, input_text))
            
        search_results = await asyncio.gather(*tasks)
        compiled_results = "\n\n".join([res.final_output for res in search_results])
        print("Web searches completed.")
        
        # 3. Write the final report
        writer_input = f"Original query: {user_query}\n\nSearch Summaries:\n{compiled_results}"
        report_result = await Runner.run(writer_agent, writer_input)
        
        print("\n=== FINAL REPORT ===\n")
        display(Markdown(report_result.final_output))
        print("\n====================\n")

In [ ]:
query = "Latest developments in football in 2025"
await run_research_workflow(query)

In [ ]:
query = "Tell me the latest gossip about AI in 2026"
await run_research_workflow(query)